# CSCI 4253 / 5253 - Lab #4 - Patent Problem with Spark DataFrames
<div>
 <h2> CSCI 4283 / 5253 
  <IMG SRC="https://www.colorado.edu/cs/profiles/express/themes/cuspirit/logo.png" WIDTH=50 ALIGN="right"/> </h2>
</div>

In [1]:
%%bash
make
# rm patents.sq3

make: Nothing to be done for 'all'.


This [Spark cheatsheet](https://s3.amazonaws.com/assets.datacamp.com/blog_assets/PySpark_SQL_Cheat_Sheet_Python.pdf) is useful as is [this reference on doing joins in Spark dataframe](http://www.learnbymarketing.com/1100/pyspark-joins-by-example/).

The [DataBricks company has one of the better reference manuals for PySpark](https://docs.databricks.com/spark/latest/dataframes-datasets/index.html) -- they show you how to perform numerous common data operations such as joins, aggregation operations following `groupBy` and the like.

In [2]:
from pyspark import SparkContext, SparkConf
from pyspark.sql import SparkSession

The following aggregation functions may be useful -- [these can be used to aggregate results of `groupby` operations](https://docs.databricks.com/spark/latest/dataframes-datasets/introduction-to-dataframes-python.html#example-aggregations-using-agg-and-countdistinct). More documentation is at the [PySpark SQL Functions manual](https://spark.apache.org/docs/2.3.0/api/python/pyspark.sql.html#module-pyspark.sql.functions). Feel free to use other functions from that library.

In [3]:
from pyspark.sql.functions import col, count, countDistinct

Create our session as described in the tutorials

In [4]:
spark = SparkSession \
    .builder \
    .appName("Lab4-Dataframe") \
    .master("local[*]")\
    .getOrCreate()

Read in the citations and patents data and check that the data makes sense. Note that unlike in the RDD solution, the data is automatically inferred to be Integer() types.

In [5]:
citations = spark.read.load('cite75_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [6]:
citations.show(5)

+-------+-------+
| CITING|  CITED|
+-------+-------+
|3858241| 956203|
|3858241|1324234|
|3858241|3398406|
|3858241|3557384|
|3858241|3634889|
+-------+-------+
only showing top 5 rows



In [7]:
patents = spark.read.load('apat63_99.txt.gz',
            format="csv", sep=",", header=True,
            compression="gzip",
            inferSchema="true")

In [8]:
patents.show(5)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+
|3070801| 1963| 1096|   NULL|     BE|   NULL|    NULL|      1|  NULL|   269|  6|    69| NULL|       1|    NULL|    0.0|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070802| 1963| 1096|   NULL|     US|     TX|    NULL|      1|  NULL|     2|  6|    63| NULL|       0|    NULL|   NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|    NULL|
|3070803| 1963| 1096|   NULL|     US|     IL|    NULL|      1|  NULL|     2|  6|    6

Create a table with Citing Patent & its state

In [9]:
citing_with_state = citations.join(
    patents.select(col("PATENT"), col("POSTATE").alias("CITING_STATE")),
    citations["CITING"] == patents["PATENT"]
).drop("PATENT")

citing_with_state.show(10)

+-------+-------+------------+
| CITING|  CITED|CITING_STATE|
+-------+-------+------------+
|3858242|1515701|          MI|
|3858242|3319261|          MI|
|3858242|3668705|          MI|
|3858242|3707004|          MI|
|3858243|2949611|        NULL|
|3858243|3146465|        NULL|
|3858243|3156927|        NULL|
|3858243|3221341|        NULL|
|3858243|3574238|        NULL|
|3858243|3681785|        NULL|
+-------+-------+------------+
only showing top 10 rows



Create a table with Cited Patent & its state

In [10]:
cited_with_state = citations.join(
    patents.select(col("PATENT"), col("POSTATE").alias("CITED_STATE")),
    citations["CITED"] == patents["PATENT"]
).drop("PATENT")

cited_with_state.show(10)

+-------+-------+-----------+
| CITING|  CITED|CITED_STATE|
+-------+-------+-----------+
|4093112|3070801|       NULL|
|4133055|3070803|         IL|
|4253313|3070803|         IL|
|4483021|3070803|         IL|
|4484363|3070803|         IL|
|4921141|3070803|         IL|
|5054122|3070803|         IL|
|5469579|3070803|         IL|
|5557807|3070803|         IL|
|5850636|3070803|         IL|
+-------+-------+-----------+
only showing top 10 rows



Join the 2 tables created above, to combine state details

In [11]:
citing_and_cited = citing_with_state.join(
    cited_with_state,
    (citing_with_state['CITING']==cited_with_state['CITING']) &
    (citing_with_state['CITED']==cited_with_state['CITED'])
).select(
    citing_with_state['CITING'],
    citing_with_state['CITED'],
    citing_with_state['CITING_STATE'],
    cited_with_state['CITED_STATE']
)

citing_and_cited.show(10)

+-------+-------+------------+-----------+
| CITING|  CITED|CITING_STATE|CITED_STATE|
+-------+-------+------------+-----------+
|3858250|3324482|          NY|         PA|
|3858252|3458874|          CA|         CA|
|3858254|3736601|          CA|       NULL|
|3858267|3626542|          NY|         FL|
|3858271|3345675|        NULL|         IL|
|3858275|3139595|          OK|         CA|
|3858276|3120030|          SC|       NULL|
|3858279|3222745|        NULL|         IL|
|3858282|3790992|          CA|       NULL|
|3858286|3286595|        NULL|       NULL|
+-------+-------+------------+-----------+
only showing top 10 rows



Filter out rows that do not have any state details & keep rows where citing state matches cited state

In [12]:
same_state = citing_and_cited.filter(
    (col("CITING_STATE") == col("CITED_STATE")) &
    col("CITING_STATE").isNotNull() &
    col("CITED_STATE").isNotNull()
)

same_state.show(10)

+-------+-------+------------+-----------+
| CITING|  CITED|CITING_STATE|CITED_STATE|
+-------+-------+------------+-----------+
|4067198|3217791|          AK|         AK|
|4107865|3841011|          AK|         AK|
|4245930|4192630|          AK|         AK|
|4742798|4180012|          AK|         AK|
|4758005|4713867|          AK|         AK|
|4843756|4184283|          AK|         AK|
|4955277|4890532|          AK|         AK|
|5140508|4727462|          AK|         AK|
|5168653|4184283|          AK|         AK|
|5172587|3706204|          AK|         AK|
+-------+-------+------------+-----------+
only showing top 10 rows



Aggregate Patent Cited counts from the same state on the Citing Patent

In [13]:
same_state_counts = same_state.groupBy("CITING") \
    .agg(count("*").alias("SAME_STATE")) \
    .withColumnRenamed("CITING", "PATENT_ID")

Perform a left join with the initial Patents table, matching on the `PATENT_ID`
- Coalesce handles the null values by imputing them with a 0

In [14]:
from pyspark.sql.functions import coalesce, lit

augmented_df = patents.join(
    same_state_counts,
    patents["PATENT"] == same_state_counts["PATENT_ID"],
    how="left"
).drop("PATENT_ID") \
 .withColumn("SAME_STATE", coalesce(col("SAME_STATE"), lit(0)))

Finally print the augmented dataframe with the same state counts in descending order of count value

In [15]:
top10 = augmented_df.orderBy(col("SAME_STATE").desc())
top10.show(10)

+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
| PATENT|GYEAR|GDATE|APPYEAR|COUNTRY|POSTATE|ASSIGNEE|ASSCODE|CLAIMS|NCLASS|CAT|SUBCAT|CMADE|CRECEIVE|RATIOCIT|GENERAL|ORIGINAL|FWDAPLAG|BCKGTLAG|SELFCTUB|SELFCTLB|SECDUPBD|SECDLWBD|SAME_STATE|
+-------+-----+-----+-------+-------+-------+--------+-------+------+------+---+------+-----+--------+--------+-------+--------+--------+--------+--------+--------+--------+--------+----------+
|5959466| 1999|14515|   1997|     US|     CA|    5310|      2|  NULL|   326|  4|    46|  159|       0|     1.0|   NULL|  0.6186|    NULL|  4.8868|  0.0455|   0.044|    NULL|    NULL|       125|
|5983822| 1999|14564|   1998|     US|     TX|  569900|      2|  NULL|   114|  5|    55|  200|       0|   0.995|   NULL|  0.7201|    NULL|   12.45|     0.0|     0.0|    NULL|    NULL|       103|
|6008204| 1999|14606|   1998| 